# 🎓 Professor's AI Assistant — Google ADK Multi-Agent

| Agent | Job |
|---|---|
| 🗂️ **Syllabus Agent** | Answers questions about your syllabus |
| 📅 **Timetable Agent** | Reads your timetable, finds free slots |
| 📝 **Quiz Agent** | Generates quiz questions and MCQs |

**Setup:** Add your Gemini key to Colab Secrets as `GEMINI_API_KEY` (🔑 icon in sidebar).
Run Cell 1, **restart runtime**, then run all remaining cells.

## Cell 1 — Install
> ⚠️ Restart runtime after this cell.

In [1]:
!pip install -q google-adk PyPDF2
print('✅ Done — restart runtime now.')

✅ Done — restart runtime now.


## Cell 2 — Imports & API Key

In [2]:
import os
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')

from google.adk.agents import LlmAgent
from google.adk.tools.agent_tool import AgentTool
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools import ToolContext
from google.genai.types import Content, Part
print('✅ Ready.')

✅ Ready.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


## Cell 3 — Upload Files (optional)
Upload your **syllabus** and/or **timetable** as PDF. Cancel to skip.

In [3]:
from google.colab import files
import PyPDF2, io

SYLLABUS_TEXT  = ''
TIMETABLE_TEXT = ''

def extract_pdf(content):
    return '\n'.join(p.extract_text() or '' for p in PyPDF2.PdfReader(io.BytesIO(content)).pages)

print('📄 Upload SYLLABUS PDF (or cancel to skip)')
try:
    up = files.upload()
    if up:
        SYLLABUS_TEXT = extract_pdf(list(up.values())[0])
        print(f'✅ Syllabus loaded ({len(SYLLABUS_TEXT)} chars)')
except: print('⚠️ Skipped.')

print('\n🗓️ Upload TIMETABLE PDF (or cancel to skip)')
try:
    up = files.upload()
    if up:
        TIMETABLE_TEXT = extract_pdf(list(up.values())[0])
        print(f'✅ Timetable loaded ({len(TIMETABLE_TEXT)} chars)')
except: print('⚠️ Skipped.')

📄 Upload SYLLABUS PDF (or cancel to skip)


Saving DEVOPS-SYLLABUS.pdf to DEVOPS-SYLLABUS (1).pdf
✅ Syllabus loaded (6772 chars)

🗓️ Upload TIMETABLE PDF (or cancel to skip)


Saving aiml_sem4_timetable (2).pdf to aiml_sem4_timetable (2) (2).pdf
✅ Timetable loaded (3165 chars)


## Cell 4 — Tools
> 📌 Tools are plain Python functions. The **docstring** is what the LLM reads to decide when to call each tool.

In [4]:
def get_syllabus(tool_context: ToolContext) -> str:
    """Return the full uploaded syllabus text."""
    text = tool_context.state.get('syllabus', SYLLABUS_TEXT)
    return text if text.strip() else 'No syllabus uploaded.'

def search_syllabus(topic: str, tool_context: ToolContext) -> str:
    """Search the syllabus for a topic or keyword.
    Args: topic: keyword to search for.
    """
    text = tool_context.state.get('syllabus', SYLLABUS_TEXT)
    if not text.strip(): return 'No syllabus uploaded.'
    matches = [l for l in text.splitlines() if topic.lower() in l.lower()]
    return ('\n'.join(matches[:20])) if matches else f"'{topic}' not found."

def get_timetable(tool_context: ToolContext) -> str:
    """Return the full timetable as structured lines (Day | Time | Subject | Professor)."""
    text = tool_context.state.get('timetable', TIMETABLE_TEXT)
    if not text.strip(): return 'No timetable uploaded.'
    lines = [l for l in text.splitlines() if ' | ' in l]
    return '\n'.join(lines) if lines else text

def search_timetable(query: str, tool_context: ToolContext) -> str:
    """Search the timetable by professor name, day, or subject.
    Use this when the user mentions a name (e.g. 'kumar', 'patel'),
    a day (e.g. 'Monday'), or a subject (e.g. 'Machine Learning').
    Each result line format: Day | Time | Subject | Professor
    Args: query: name, day, or subject to search for.
    """
    text = tool_context.state.get('timetable', TIMETABLE_TEXT)
    if not text.strip(): return 'No timetable uploaded.'
    lines = [l for l in text.splitlines() if ' | ' in l]
    matches = [l for l in lines if query.lower() in l.lower()]
    return '\n'.join(matches) if matches else f"No entries found for '{query}'."


def save_quiz(title: str, content: str, tool_context: ToolContext) -> str:
    """Save a generated quiz to session memory.
    Args: title: quiz title. content: full quiz text.
    """
    quizzes = tool_context.state.get('quizzes', [])
    quizzes.append({'title': title, 'content': content})
    tool_context.state['quizzes'] = quizzes
    return f"Saved '{title}'. Total: {len(quizzes)}."

def list_quizzes(tool_context: ToolContext) -> str:
    """List all quizzes saved in this session."""
    q = tool_context.state.get('quizzes', [])
    return '\n'.join(f"{i+1}. {x['title']}" for i,x in enumerate(q)) if q else 'No quizzes saved.'

print('✅ Tools defined.')

✅ Tools defined.


## Cell 5 — Agents
> 📌 Each agent's **`description`** tells the orchestrator when to delegate to it.

In [5]:
MODEL = 'gemini-2.5-flash'

syllabus_agent = LlmAgent(
    name='syllabus_agent',
    model=MODEL,
    description='Answers questions about the course syllabus — topics, weeks, content.',
    instruction='Use get_syllabus() or search_syllabus(topic) to answer.',
    tools=[get_syllabus, search_syllabus],
)

timetable_agent = LlmAgent(
    name='timetable_agent',
    model=MODEL,
    description='Answers schedule questions — class times, professor schedules, free slots. Also handles queries by professor name.',
    instruction=(
        'Answer schedule questions using search_timetable(query).\n'
        'If user mentions a name like "kumar", search by that name.\n'
        'If user mentions a day like "Monday", search by that day.\n'
        'The time slots in a day are: 9:00-10:00, 10:00-11:00, 11:00-12:00, '
        '12:00-1:00 (lunch), 2:00-3:00, 3:00-4:00, 4:00-5:00.\n'
        'If asked for free time or office hours, first find the busy slots for that professor, '
        'then list the remaining slots as free. Always show both busy and free slots.'
    ),
    tools=[get_timetable, search_timetable],
)

quiz_agent = LlmAgent(
    name='quiz_agent',
    model=MODEL,
    description='Generates quiz questions and MCQs on any topic or from the syllabus.',
    instruction='Generate questions with answer keys. Save each quiz with save_quiz().',
    tools=[save_quiz, list_quizzes, get_syllabus],
)

# Orchestrator — wraps sub-agents as AgentTool objects
root_agent = LlmAgent(
    name='professor_assistant',
    model=MODEL,
    description="Professor's assistant for syllabus, timetable, and quiz tasks.",
    instruction=(
        'Route to the right agent:\n'
        '- syllabus_agent : syllabus content, topics\n'
        '- timetable_agent: schedule, free slots\n'
        '- quiz_agent     : generate quizzes, MCQs'
    ),
    tools=[
        AgentTool(agent=syllabus_agent),
        AgentTool(agent=timetable_agent),
        AgentTool(agent=quiz_agent),
    ],
)
print('✅ Agents ready.')

✅ Agents ready.


## Cell 6 — Runner, Session & ask()
> `ask()` shows every delegation step (→ agent called) and the final answer.

In [6]:
APP_NAME = 'prof_assistant'
USER_ID  = 'professor'

session_service = InMemorySessionService()
runner  = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)
session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID)

if SYLLABUS_TEXT:  session.state['syllabus']  = SYLLABUS_TEXT
if TIMETABLE_TEXT: session.state['timetable'] = TIMETABLE_TEXT

def ask(question):
    """Send a question. Prints each agent delegation step and the final answer."""
    print(f'\n🧑\u200d🏫 {question}\n' + '─'*55)
    msg = Content(role='user', parts=[Part(text=question)])
    for event in runner.run(user_id=USER_ID, session_id=session.id, new_message=msg):
        if not event.content: continue
        for part in event.content.parts:
            if hasattr(part, 'function_call') and part.function_call:
                print(f'  → [{part.function_call.name}]')
            elif hasattr(part, 'text') and part.text and event.is_final_response():
                print(part.text.strip())

print('✅ Ready.')

✅ Ready.


---
## Demo — Syllabus

In [13]:
ask('Summarise the course syllabus.')


🧑‍🏫 Summarise the course syllabus.
───────────────────────────────────────────────────────
  → [syllabus_agent]
This course, "DEVOPS (BCSL657D)," is a 6th-semester practical course worth 1 credit. It aims to introduce students to DevOps terminology, version control tools (like Git), Continuous Integration/Testing/Deployment (CI/CT/CD) concepts, and configuration management using Ansible. A key focus is on demonstrating cloud-based DevOps tools, specifically Azure DevOps, to solve real-world problems.

The syllabus is structured around 12 experiments:
*   **Build Automation:** Introduction to and practical work with Maven and Gradle.
*   **Continuous Integration:** Setting up and using Jenkins for CI pipelines, integrating with Maven/Gradle, and running automated builds and tests.
*   **Configuration Management:** Basics and practical application of Ansible for automating server configurations.
*   **Integrated Pipelines:** Combining Jenkins and Ansible for CI and artifact deployment.


In [8]:
ask('Is github covered? Which section?')


🧑‍🏫 Is github covered? Which section?
───────────────────────────────────────────────────────
  → [syllabus_agent]
Yes, GitHub is covered under "Integrating Code Repositories".


## Demo — Timetable

In [14]:
ask('I am Dr. Rajesh Kumar. What is my schedule on Monday?')


🧑‍🏫 I am Dr. Rajesh Kumar. What is my schedule on Monday?
───────────────────────────────────────────────────────
  → [timetable_agent]
I could not find any schedule for Dr. Rajesh Kumar on Monday.


In [10]:
ask('When do I have free time for office hours?')


🧑‍🏫 When do I have free time for office hours?
───────────────────────────────────────────────────────
  → [timetable_agent]
Please specify a professor's name so I can check their schedule for office hours.


## Demo — Quiz

In [11]:
ask('Create a 2-question MCQ quiz based on the devops syllabus.')


🧑‍🏫 Create a 2-question MCQ quiz based on the devops syllabus.
───────────────────────────────────────────────────────
  → [quiz_agent]
Here is your 2-question MCQ quiz based on the DevOps syllabus:

**DevOps Syllabus MCQ Quiz**

1.  Which of the following tools is primarily used for Continuous Integration in a DevOps pipeline, as per the syllabus?
    A) Git
    B) Ansible
    C) Jenkins
    D) Maven

    **Answer: C) Jenkins**

2.  What is the primary purpose of Ansible in a DevOps context, according to the syllabus?
    A) Version control
    B) Build automation
    C) Configuration management
    D) Cloud deployment

    **Answer: C) Configuration management**

I have saved this quiz with the title "DevOps Syllabus MCQ Quiz".


## Demo — Multi-agent chaining
One question, two agents called in sequence.

In [12]:
ask('Read my syllabus and create a quiz on the first topic.')


🧑‍🏫 Read my syllabus and create a quiz on the first topic.
───────────────────────────────────────────────────────
  → [syllabus_agent]
  → [quiz_agent]
I have created a quiz on "Introduction to Maven and Gradle: Overview of Build Automation Tools, Key Differences Between Maven and Gradle, Installation and Setup" and saved it as "Introduction to Maven and Gradle Quiz".
